### Introduction and Imports

Import the required libraries and display the initial environment information.

In [28]:
print("Hello, Image Steganography!")

import numpy as np
from PIL import Image
import zlib

print("NumPy version:", np.__version__)
print("Pillow is ready!")

Hello, Image Steganography!
NumPy version: 2.4.2
Pillow is ready!


### Load Utility Functions

Import and reload the helper functions and protocol constants from `utils.py`.

In [30]:
import sys
import importlib

sys.path.append("..")

import utils
importlib.reload(utils)

from utils import (
    prepare_image,
    message_to_bytes,
    message_to_bits,
    bits_to_bytes,
    create_header,
    check_capacity,
    SEQUENTIAL,
    PRNG,
    HEADER_BITS
)

### Select Cover Image

Open a file dialog to select the PNG cover image.

In [31]:
import tkinter as tk
from tkinter import filedialog

root = tk.Tk()
root.withdraw()

image_path = filedialog.askopenfilename(
    title="Select a Cover Image",
    filetypes=[("PNG Images", "*.png")]
)

root.destroy()

if not image_path:
    raise ValueError("No image selected.")

try:
    with Image.open(image_path) as img:
        if img.format != "PNG":
            raise ValueError("Cover image must be in PNG format.")

        pixels = prepare_image(img)

except Image.UnidentifiedImageError:
    raise ValueError("The selected file is not a valid image.")

print("Image loaded successfully!")
print("Image shape:", pixels.shape)
print("Image type:", pixels.dtype)
print("Selected file:", image_path)

Image loaded successfully!
Image shape: (508, 728, 3)
Image type: uint8
Selected file: C:/Users/silent-repair/Desktop/bahr/pro/cut the rope/v4/images/candy.png


### Enter Secret Message

Receive the secret message and validate that it is not empty.

In [32]:
message = input("Enter the secret message: ")

if not message:
    raise ValueError("Secret message cannot be empty.")

message_bytes = message_to_bytes(message)
payload_bits = message_to_bits(message)

print("Message:", message)
print("Message bytes:", message_bytes)
print("Message length (bytes):", len(message_bytes))
print("Message length (bits):", len(payload_bits))

Message: gooodbyeee
Message bytes: b'gooodbyeee'
Message length (bytes): 10
Message length (bits): 80


### Convert Message to UTF-8 Bits

Convert the secret message into UTF-8 bytes and binary bits.

In [34]:
header_bits = create_header(message_bytes, SEQUENTIAL)

print("Header bits:", header_bits)
print("Header length:", len(header_bits))

Header bits: 0101001101010100010001010100011100000010000000000000000000000000000000000000101000101010101100000011001010011000
Header length: 112


### Create Sequential Header

Create the V2 header using the Sequential method identifier.

In [35]:
full_payload_bits = header_bits + payload_bits

print("Header length:", len(header_bits))
print("Message length:", len(payload_bits))
print("Total payload length:", len(full_payload_bits))

Header length: 112
Message length: 80
Total payload length: 192


### Validate Header and Image Capacity

Verify the header fields, CRC32 value, message length, and available image capacity.

In [42]:
assert len(header_bits) == HEADER_BITS
assert len(header_bits) == 112

assert len(payload_bits) == len(message_bytes) * 8

assert len(full_payload_bits) == HEADER_BITS + len(payload_bits)

header_bytes = bits_to_bytes(header_bits)

assert header_bytes[:4] == b"STEG"
assert header_bytes[4] == 2
assert header_bytes[5] == SEQUENTIAL

header_length = int.from_bytes(header_bytes[6:10], "big")
assert header_length == len(message_bytes)

header_crc = int.from_bytes(header_bytes[10:14], "big")
expected_crc = zlib.crc32(message_bytes) & 0xFFFFFFFF

assert header_crc == expected_crc

available_bits = check_capacity(pixels, payload_bits)

print("All preliminary checks passed!")

print("Header size:", len(header_bits), "bits")
print("Message size:", len(payload_bits), "bits")
print("Total required:", len(full_payload_bits), "bits")
print("Available payload capacity:", available_bits, "bits")

All preliminary checks passed!
Header size: 112 bits
Message size: 80 bits
Total required: 192 bits
Available payload capacity: 1109360 bits


### Prepare Image Data

Convert the image data into a one-dimensional array
and create a separate copy for embedding.

In [43]:
values = pixels.flatten().copy()
stego_values = values.copy()

print("Image data prepared successfully!")
print("Original array size:", values.size)
print("Stego array size:", stego_values.size)

Image data prepared successfully!
Original array size: 1109472
Stego array size: 1109472


### Embed Header

Embed the 112-bit V2 header into the first 112
color channel values using the Least Significant Bit (LSB).

In [45]:
for i, bit in enumerate(header_bits):
    stego_values[i] = (int(stego_values[i]) & 254) | int(bit)

print("Header embedded successfully!")
print("Header bits embedded:", len(header_bits))

Header embedded successfully!
Header bits embedded: 112


### Embed Message - Sequential

Embed the secret message bits sequentially,
starting from channel position 112.

In [46]:
for i, bit in enumerate(payload_bits):
    position = HEADER_BITS + i
    stego_values[position] = (
        (int(stego_values[position]) & 254) | int(bit)
    )

print("Message embedded successfully!")
print("Embedding method: Sequential")
print("Message bits embedded:", len(payload_bits))

Message embedded successfully!
Embedding method: Sequential
Message bits embedded: 80


### Verify Sequential Embedding

Verify that the embedded header and message bits
match the original data.

In [47]:
embedded_header = ''.join(
    str(int(value) & 1)
    for value in stego_values[:HEADER_BITS]
)

embedded_message = ''.join(
    str(int(value) & 1)
    for value in stego_values[
        HEADER_BITS:HEADER_BITS + len(payload_bits)
    ]
)

assert embedded_header == header_bits
assert embedded_message == payload_bits

print("Sequential embedding verification passed!")
print("Header verification: Successful")
print("Message verification: Successful")

Sequential embedding verification passed!
Header verification: Successful
Message verification: Successful


### Embed Message - PRNG

Embed the secret message bits at pseudo-random
positions using the V2 protocol and a fixed seed.

In [69]:
import random

prng_values = values.copy()

prng_header_bits = create_header(message_bytes, PRNG)

for i, bit in enumerate(prng_header_bits):
    prng_values[i] = (
        (int(prng_values[i]) & 254) | int(bit)
    )

seed = 12345

positions = random.Random(seed).sample(
    range(HEADER_BITS, len(prng_values)),
    len(payload_bits)
)

for position, bit in zip(positions, payload_bits):
    prng_values[position] = (
        (int(prng_values[position]) & 254) | int(bit)
    )

print("PRNG embedding completed!")
print("Embedding method: PRNG")
print("Seed:", seed)
print("Message bits embedded:", len(payload_bits))

PRNG embedding completed!
Embedding method: PRNG
Seed: 12345
Message bits embedded: 80


### Verify PRNG Embedding

Verify that the embedded header and message bits
match the original data at their selected positions.

In [68]:
embedded_header = ''.join(
    str(int(value) & 1)
    for value in prng_values[:HEADER_BITS]
)

embedded_message = ''.join(
    str(int(prng_values[position]) & 1)
    for position in positions
)

assert embedded_header == prng_header_bits
assert embedded_message == payload_bits

embedded_header_bytes = bits_to_bytes(embedded_header)
assert embedded_header_bytes[5] == PRNG

print("PRNG embedding verification passed!")
print("Header verification: Successful")
print("Message verification: Successful")
print("Method verification: PRNG (1)")

PRNG embedding verification passed!
Header verification: Successful
Message verification: Successful
Method verification: PRNG (1)


### Save Stego Images

Reshape and save the Sequential and PRNG embedded images as PNG files.

In [67]:
import os

sequential_pixels = stego_values.reshape(pixels.shape)
prng_pixels = prng_values.reshape(pixels.shape)

sequential_path = os.path.join("..", "stego_sequential.png")
prng_path = os.path.join("..", "stego_prng.png")

Image.fromarray(sequential_pixels).save(
    sequential_path, format="PNG"
)

Image.fromarray(prng_pixels).save(
    prng_path, format="PNG"
)

print("Stego images saved successfully!")
print("Sequential image:", os.path.abspath(sequential_path))
print("PRNG image:", os.path.abspath(prng_path))

Stego images saved successfully!
Sequential image: c:\Users\silent-repair\Documents\image-steganography\stego_sequential.png
PRNG image: c:\Users\silent-repair\Documents\image-steganography\stego_prng.png


### Verify Saved Stego Images

Reopen both PNG files and verify that the embedded data remains unchanged after saving.

In [53]:
with Image.open(sequential_path) as img:
    saved_sequential = np.array(img)
    assert img.format == "PNG"
    assert img.mode == "RGB"

with Image.open(prng_path) as img:
    saved_prng = np.array(img)
    assert img.format == "PNG"
    assert img.mode == "RGB"

assert np.array_equal(saved_sequential, sequential_pixels)
assert np.array_equal(saved_prng, prng_pixels)

print("Saved image verification passed!")
print("Sequential image: Verified")
print("PRNG image: Verified")

Saved image verification passed!
Sequential image: Verified
PRNG image: Verified


### Test UTF-8 Message Conversion

Verify that the user-provided message is correctly converted into UTF-8 bytes and binary bits.

In [54]:
test_bytes = message_to_bytes(message)
test_bits = message_to_bits(message)

print("Original message:", message)
print("UTF-8 bytes:", len(test_bytes))
print("Message bits:", len(test_bits))

assert len(test_bits) == len(test_bytes) * 8

print("UTF-8 conversion test passed!")

Original message: gooodbyeee
UTF-8 bytes: 10
Message bits: 80
UTF-8 conversion test passed!


### Test Image Capacity with UTF-8 Message

Verify that the image capacity check correctly handles messages containing UTF-8 characters.

In [56]:
check_capacity(pixels, test_bits)

print("UTF-8 message capacity test passed!")
print("Required bits:", HEADER_BITS + len(test_bits))
print("Available channels:", pixels.size)

UTF-8 message capacity test passed!
Required bits: 192
Available channels: 1109472


### Test UTF-8 Message Embedding

Embed the Persian UTF-8 message using both Sequential and PRNG methods and verify the embedded bits.

In [57]:
utf8_header_sequential = create_header(test_bytes, SEQUENTIAL)
utf8_header_prng = create_header(test_bytes, PRNG)

utf8_sequential_values = pixels.flatten().copy()
utf8_prng_values = pixels.flatten().copy()

for i, bit in enumerate(utf8_header_sequential):
    utf8_sequential_values[i] = (
        (int(utf8_sequential_values[i]) & 254) | int(bit)
    )

for i, bit in enumerate(test_bits):
    position = HEADER_BITS + i
    utf8_sequential_values[position] = (
        (int(utf8_sequential_values[position]) & 254) | int(bit)
    )

for i, bit in enumerate(utf8_header_prng):
    utf8_prng_values[i] = (
        (int(utf8_prng_values[i]) & 254) | int(bit)
    )

utf8_positions = random.Random(seed).sample(
    range(HEADER_BITS, len(utf8_prng_values)),
    len(test_bits)
)

for position, bit in zip(utf8_positions, test_bits):
    utf8_prng_values[position] = (
        (int(utf8_prng_values[position]) & 254) | int(bit)
    )

print("UTF-8 message embedding completed!")
print("Sequential method: Completed")
print("PRNG method: Completed")

UTF-8 message embedding completed!
Sequential method: Completed
PRNG method: Completed


### Verify UTF-8 Message Embedding

Verify that the Persian message bits and method identifiers are correctly embedded using both Sequential and PRNG methods.

In [58]:
embedded_seq_header = ''.join(
    str(int(value) & 1)
    for value in utf8_sequential_values[:HEADER_BITS]
)

embedded_seq_message = ''.join(
    str(int(value) & 1)
    for value in utf8_sequential_values[
        HEADER_BITS:HEADER_BITS + len(test_bits)
    ]
)

assert embedded_seq_header == utf8_header_sequential
assert embedded_seq_message == test_bits
assert bits_to_bytes(embedded_seq_header)[5] == SEQUENTIAL


embedded_prng_header = ''.join(
    str(int(value) & 1)
    for value in utf8_prng_values[:HEADER_BITS]
)

embedded_prng_message = ''.join(
    str(int(utf8_prng_values[position]) & 1)
    for position in utf8_positions
)

assert embedded_prng_header == utf8_header_prng
assert embedded_prng_message == test_bits
assert bits_to_bytes(embedded_prng_header)[5] == PRNG

print("UTF-8 embedding verification passed!")
print("Sequential verification: Successful")
print("PRNG verification: Successful")
print("Method identifiers: Correct")

UTF-8 embedding verification passed!
Sequential verification: Successful
PRNG verification: Successful
Method identifiers: Correct


### Test UTF-8 Message Recovery

Recover the embedded Persian message bits and decode them back into the original UTF-8 text.

In [59]:
recovered_seq_bytes = bits_to_bytes(embedded_seq_message)
recovered_seq_message = recovered_seq_bytes.decode("utf-8")

recovered_prng_bytes = bits_to_bytes(embedded_prng_message)
recovered_prng_message = recovered_prng_bytes.decode("utf-8")

assert recovered_seq_message == message
assert recovered_prng_message == message

print("UTF-8 message recovery passed!")
print("Original message:", message)
print("Sequential recovered:", recovered_seq_message)
print("PRNG recovered:", recovered_prng_message)

UTF-8 message recovery passed!
Original message: gooodbyeee
Sequential recovered: gooodbyeee
PRNG recovered: gooodbyeee


### Test UTF-8 CRC32 Validation

Verify that the CRC32 checksum stored in each header matches the original Persian message.

In [60]:
seq_header_bytes = bits_to_bytes(embedded_seq_header)
prng_header_bytes = bits_to_bytes(embedded_prng_header)

seq_stored_crc = int.from_bytes(seq_header_bytes[10:14], byteorder="big")
prng_stored_crc = int.from_bytes(prng_header_bytes[10:14], byteorder="big")

calculated_crc = zlib.crc32(test_bytes) & 0xFFFFFFFF

assert seq_stored_crc == calculated_crc
assert prng_stored_crc == calculated_crc

print("UTF-8 CRC32 validation passed!")
print("Sequential CRC32: Verified")
print("PRNG CRC32: Verified")
print("Calculated CRC32:", hex(calculated_crc))

UTF-8 CRC32 validation passed!
Sequential CRC32: Verified
PRNG CRC32: Verified
Calculated CRC32: 0x2ab03298


### Test Empty Message Validation

Verify that the embedding module correctly rejects empty secret messages.

In [62]:
empty_message = ""

try:
    if not empty_message:
        raise ValueError("Secret message cannot be empty.")

except ValueError as error:
    print("Empty message rejected successfully!")
    print("Error:", error)

Empty message rejected successfully!
Error: Secret message cannot be empty.


### Test Long Message Capacity

Verify that the embedding module correctly checks image capacity for longer secret messages.

In [63]:
long_message = message * 20

long_bytes = message_to_bytes(long_message)
long_bits = message_to_bits(long_message)

check_capacity(pixels, long_bits)

print("Long message capacity test passed!")
print("Message length (bytes):", len(long_bytes))
print("Message length (bits):", len(long_bits))
print("Required bits:", HEADER_BITS + len(long_bits))
print("Available channels:", pixels.size)

Long message capacity test passed!
Message length (bytes): 200
Message length (bits): 1600
Required bits: 1712
Available channels: 1109472


### Test Long Message Embedding

Embed a longer secret message using both Sequential and PRNG methods and verify successful recovery.

In [64]:
long_header_sequential = create_header(long_bytes, SEQUENTIAL)
long_header_prng = create_header(long_bytes, PRNG)

long_sequential_values = pixels.flatten().copy()
long_prng_values = pixels.flatten().copy()

for i, bit in enumerate(long_header_sequential):
    long_sequential_values[i] = (
        (int(long_sequential_values[i]) & 254) | int(bit)
    )

for i, bit in enumerate(long_bits):
    position = HEADER_BITS + i
    long_sequential_values[position] = (
        (int(long_sequential_values[position]) & 254) | int(bit)
    )

for i, bit in enumerate(long_header_prng):
    long_prng_values[i] = (
        (int(long_prng_values[i]) & 254) | int(bit)
    )

long_positions = random.Random(seed).sample(
    range(HEADER_BITS, len(long_prng_values)),
    len(long_bits)
)

for position, bit in zip(long_positions, long_bits):
    long_prng_values[position] = (
        (int(long_prng_values[position]) & 254) | int(bit)
    )

print("Long message embedding completed!")
print("Sequential method: Completed")
print("PRNG method: Completed")

Long message embedding completed!
Sequential method: Completed
PRNG method: Completed


### Verify Long Message Embedding

Verify that the complete long message is correctly embedded and recovered using both methods.

In [65]:
long_seq_embedded_bits = ''.join(
    str(int(value) & 1)
    for value in long_sequential_values[
        HEADER_BITS:HEADER_BITS + len(long_bits)
    ]
)

long_prng_embedded_bits = ''.join(
    str(int(long_prng_values[position]) & 1)
    for position in long_positions
)

long_seq_recovered = bits_to_bytes(
    long_seq_embedded_bits
).decode("utf-8")

long_prng_recovered = bits_to_bytes(
    long_prng_embedded_bits
).decode("utf-8")

assert long_seq_recovered == long_message
assert long_prng_recovered == long_message

print("Long message embedding verification passed!")
print("Sequential recovery: Successful")
print("PRNG recovery: Successful")
print("Message integrity: Verified")

Long message embedding verification passed!
Sequential recovery: Successful
PRNG recovery: Successful
Message integrity: Verified


### Test Insufficient Image Capacity

Verify that the embedding module rejects messages that exceed the available image capacity.

In [71]:
oversized_bits = "0" * (pixels.size - HEADER_BITS + 1)

try:
    check_capacity(pixels, oversized_bits)
    raise AssertionError("Oversized message was not rejected!")

except ValueError as error:
    print("Insufficient capacity test passed!")
    print("Error:", error)

Insufficient capacity test passed!
Error: Insufficient image capacity! Required: 1109473 bits, Available: 1109472 bits.
